In [0]:
from pyspark.sql import SparkSession  # -> contains the dataframe andsql interfaces
from pyspark.sql import functions as F # -> contains spark column functions
from pyspark.sql.window import Window # -> contains window functions

spark = ( # -> it is not a dataframe but a spark session used to create/read dataframes n run sql
    SparkSession.builder
    .appName("PySparkCourse")
    .getOrCreate() #-> create a session or load existing one
)

print(spark.version)

4.2.0


# **INSPECTING DATAFRAMES**

calling the csv files from the Volumes and loading it

In [0]:
orders_path = "/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders.csv"
customers_path = "/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/customers.csv"
products_path = "/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/products.csv"
employees_path = "/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/employees.csv"
text_path = "/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/order_lines.csv"

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
)

customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
)

employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(employees_path)
)


Calling tables and creating a dataframe of it

In [0]:
%skip
orders_path = "pyspark_crash_course.pyspark_schema.orders"
customers_path = "pyspark_crash_course.pyspark_schema.customers"
products_path = "pyspark_crash_course.pyspark_schema.products"
employees_path = "pyspark_crash_course.pyspark_schema.employees"

orders = spark.read.table(orders_path)
customers = spark.read.table(customers_path)
products = spark.read.table(products_path)
employees = spark.read.table(employees_path)

# orders.show()

In [0]:
%skip
orders.show()

In [0]:
orders.show()

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|
|    1010|   Kiran| North|Monitor|       1|     220.0|2026-01-11|
|    1011|  Fatima|  West|Monitor|       2|     220.0|2026-01-12|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1013|

In [0]:
orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
print(orders.columns)

['order_id', 'customer', 'region', 'product', 'quantity', 'unit_price', 'order_date']


In [0]:
print(orders.dtypes)

[('order_id', 'int'), ('customer', 'string'), ('region', 'string'), ('product', 'string'), ('quantity', 'int'), ('unit_price', 'double'), ('order_date', 'date')]


In [0]:
orders.describe("quantity", "unit_price").show()

+-------+------------------+-----------------+
|summary|          quantity|       unit_price|
+-------+------------------+-----------------+
|  count|                61|               61|
|   mean|2.9672131147540983|701.9672131147541|
| stddev|1.4372104127412253|494.0000663702084|
|    min|                 1|            220.0|
|    max|                 5|           1200.0|
+-------+------------------+-----------------+



In [0]:
# orders.summary("count","min","25%","mean","max").show()
orders.summary().show()

+-------+------------------+--------+------+-------+------------------+-----------------+
|summary|          order_id|customer|region|product|          quantity|       unit_price|
+-------+------------------+--------+------+-------+------------------+-----------------+
|  count|                61|      61|    57|     61|                61|               61|
|   mean|1030.1639344262296|    NULL|  NULL|   NULL|2.9672131147540983|701.9672131147541|
| stddev| 17.51587882262725|    NULL|  NULL|   NULL|1.4372104127412253|494.0000663702084|
|    min|              1001|    Asha|  East| Laptop|                 1|            220.0|
|    25%|              1015|    NULL|  NULL|   NULL|                 2|            220.0|
|    50%|              1030|    NULL|  NULL|   NULL|                 3|            220.0|
|    75%|              1045|    NULL|  NULL|   NULL|                 4|           1200.0|
|    max|              1060|    Lina|  West|Monitor|                 5|           1200.0|
+-------+-

In [0]:
filtered = orders.filter(F.col("quantity") >= 2).select("order_id","product","quantity")
# filtered.explain()
filtered.explain(True)

== Parsed Logical Plan ==
'Project ['order_id, 'product, 'quantity]
+- 'Filter '`>=`('quantity, 2)
   +- Relation [order_id#11110,customer#11111,region#11112,product#11113,quantity#11114,unit_price#11115,order_date#11116] csv

== Analyzed Logical Plan ==
order_id: int, product: string, quantity: int
Project [order_id#11110, product#11113, quantity#11114]
+- Filter (quantity#11114 >= 2)
   +- Relation [order_id#11110,customer#11111,region#11112,product#11113,quantity#11114,unit_price#11115,order_date#11116] csv

== Optimized Logical Plan ==
Project [order_id#11110, product#11113, quantity#11114]
+- Filter (isnotnull(quantity#11114) AND (quantity#11114 >= 2))
   +- Relation [order_id#11110,customer#11111,region#11112,product#11113,quantity#11114,unit_price#11115,order_date#11116] csv

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonFilter (isnotnull(quantity#11114) AND (quantity#11114 >= 2))
      +- PhotonRowToColumnar
         +- FileScan csv [order_id#11110,pr

# **Schemas and Data Types**

### 1. Inferred Schema

In [0]:
inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
)


inferred.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: date (nullable = true)



### 2. Explicit -> StructType / StructField

In [0]:

from pyspark.sql.types import *

schema = StructType([
    StructField("order_id",IntegerType(),nullable = False),
    StructField("customer",StringType(),nullable = False),
    StructField("region",StringType(),nullable = False),
    StructField("product",StringType(),nullable = False),
    StructField("quantity",IntegerType(),nullable = False),
    StructField("unit_price",DoubleType(),nullable = False),
    StructField("order_date",StringType(),nullable = False) #-> mmyyddhhmmss
])

orders_explicit = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(orders_path)
)

orders_explicit.printSchema()



root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)



In [0]:
orders_explicit.show()

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|
|    1010|   Kiran| North|Monitor|       1|     220.0|2026-01-11|
|    1011|  Fatima|  West|Monitor|       2|     220.0|2026-01-12|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1013|

### 3. DDL schema strings

In [0]:
ddl_schema = "order_id INT, customer STRING, region STRING, product STRING, quantity INT, unit_price DOUBLE, order_date STRING"

orders_ddl = (
    spark.read
    .option("header", True)
    .schema(ddl_schema)
    .csv(orders_path)
)

orders_ddl.printSchema()
orders_ddl.show()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200

### 4. casting and parsing dates

In [0]:
typed = (
    orders
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
)

typed.printSchema()

typed.select("order_id","customer","region","product","quantity","unit_price","order_date").show()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0

# Reading & Writing Data

### 1.Read CSV

In [0]:
csv_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema",True)
    .option("sep", ",")
    .option("mode", "PERMISSIVE")
    .load(orders_path)
)
#  Equiavalent shorthand
csv_df = spark.read.option("header",True).csv(orders_path)
csv_df.show()

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|
|    1010|   Kiran| North|Monitor|       1|     220.0|2026-01-11|
|    1011|  Fatima|  West|Monitor|       2|     220.0|2026-01-12|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1013|

### 2. Read JSON and Parquet

In [0]:
json_df = spark.read.json("Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders.json")
parquet_df = spark.read.json("Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders.parquet")

### 3. Write data and modes

In [0]:
output_path = "Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders"
(
    typed.write
    .mode("overwrite")
    .format("parquet")
    .save(output_path)
)


+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|
|    1010|   Kiran| North|Monitor|       1|     220.0|2026-01-11|
|    1011|  Fatima|  West|Monitor|       2|     220.0|2026-01-12|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1013|

### 4. Partitioned output

In [0]:
(
    typed.write.mode("overwrite")
    .partitionBy("region")
    .parquet("Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders_by_region")
)

In [0]:
display(orders)

order_id,customer,region,product,quantity,unit_price,order_date
1001,Hana,South,Laptop,2,1200.0,2026-01-02
1002,Chen,North,Laptop,3,1200.0,2026-01-03
1003,Jia,West,Monitor,4,220.0,2026-01-04
1004,Eli,East,Monitor,5,220.0,2026-01-05
1005,Lina,South,Monitor,1,220.0,2026-01-06
1006,Gabe,North,Laptop,2,1200.0,2026-01-07
1007,Ben,West,Laptop,3,1200.0,2026-01-08
1008,Ishan,East,Laptop,4,1200.0,2026-01-09
1009,Divya,South,Monitor,5,220.0,2026-01-10
1010,Kiran,North,Monitor,1,220.0,2026-01-11


In [0]:
reg = spark.read.parquet("/Volumes/pyspark_crash_course/pyspark_schema/pyspark_volume/orders_by_region/region=East/")
display(reg)

order_id,customer,product,quantity,unit_price,order_date
1060,Asha,Laptop,1,1200.0,2026-01-05
1056,Ishan,Laptop,2,1200.0,2026-01-01
1048,Asha,Laptop,4,1200.0,2026-01-21
1044,Ishan,Laptop,5,1200.0,2026-01-17
1040,Eli,Monitor,1,220.0,2026-01-13
1036,Asha,Laptop,2,1200.0,2026-01-09
1032,Ishan,Laptop,3,1200.0,2026-01-05
1028,Eli,Monitor,4,220.0,2026-01-01
1024,Asha,Laptop,5,1200.0,2026-01-25
1020,Ishan,Laptop,1,1200.0,2026-01-21


# Selecting and Creating Columns

### 1. select() , col(), aliases, and aritmetic

In [0]:
# typed.select("order_id","customer","product").show()
products.select("product","unit_price").show()

+----------+----------+
|   product|unit_price|
+----------+----------+
|    Laptop|    1200.0|
|     Mouse|      25.0|
|  Keyboard|      75.0|
|   Monitor|     220.0|
|Headphones|      85.0|
|    Webcam|      60.0|
+----------+----------+



In [0]:
typed.select(
    F.col("order_id"),
    F.col("customer").alias("buyer"),
    (F.col("quantity") * F.col("unit_price")).alias("line_total")
).show(5)

+--------+-----+----------+
|order_id|buyer|line_total|
+--------+-----+----------+
|    1001| Hana|    2400.0|
|    1002| Chen|    3600.0|
|    1003|  Jia|     880.0|
|    1004|  Eli|    1100.0|
|    1005| Lina|     220.0|
+--------+-----+----------+
only showing top 5 rows


### 2. selectExpr

In [0]:
typed.selectExpr(
    "order_id",
    "customer AS buyer",
    "quantity * unit_price AS line_total"
).show(5)

+--------+-----+----------+
|order_id|buyer|line_total|
+--------+-----+----------+
|    1001| Hana|    2400.0|
|    1002| Chen|    3600.0|
|    1003|  Jia|     880.0|
|    1004|  Eli|    1100.0|
|    1005| Lina|     220.0|
+--------+-----+----------+
only showing top 5 rows


### 3. lit(), withColumn(), rename, drop, cast

In [0]:
enriched = (
    typed
    .withColumn("line_total",F.col("quantity")*F.col("unit_price"))
    .withColumn("currency", F.lit("INR"))
    .withColumn("customer_name", F.col("customer"))
    .drop("customer")
    .withColumn("order_id",F.col("order_id").cast("long"))
)

enriched.show()

+--------+------+-------+--------+----------+----------+----------+--------+-------------+
|order_id|region|product|quantity|unit_price|order_date|line_total|currency|customer_name|
+--------+------+-------+--------+----------+----------+----------+--------+-------------+
|    1001| South| Laptop|       2|    1200.0|2026-01-02|    2400.0|     INR|         Hana|
|    1002| North| Laptop|       3|    1200.0|2026-01-03|    3600.0|     INR|         Chen|
|    1003|  West|Monitor|       4|     220.0|2026-01-04|     880.0|     INR|          Jia|
|    1004|  East|Monitor|       5|     220.0|2026-01-05|    1100.0|     INR|          Eli|
|    1005| South|Monitor|       1|     220.0|2026-01-06|     220.0|     INR|         Lina|
|    1006| North| Laptop|       2|    1200.0|2026-01-07|    2400.0|     INR|         Gabe|
|    1007|  West| Laptop|       3|    1200.0|2026-01-08|    3600.0|     INR|          Ben|
|    1008|  East| Laptop|       4|    1200.0|2026-01-09|    4800.0|     INR|        Ishan|

In [0]:
typed.withColumnRenamed("unit_price", "price_each").show(2)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|price_each|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
+--------+--------+------+-------+--------+----------+----------+
only showing top 2 rows


# Filtering, Sorting and Conditional Logic

### 1. filter()/where(), comparisons, membership

In [0]:
typed.filter(F.col("quantity") > 2).show()

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1013|    Hana|  NULL| Laptop|       4|    1200.0|2026-01-14|
|    1014|    Chen| North| Laptop|       5|    1200.0|2026-01-15|
|    1017|    Lina| South|Monitor|       3|     220.0|2026-01-18|
|    1018|    Gabe| North| Laptop|       4|    1200.0|2026-01-19|
|    1019|     Ben|  West| Laptop|       5|    1200.0|2026-01-20|
|    1022|

In [0]:
typed.filter(F.col("region") == "East").show()

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1016|     Eli|  East|Monitor|       2|     220.0|2026-01-17|
|    1020|   Ishan|  East| Laptop|       1|    1200.0|2026-01-21|
|    1024|    Asha|  East| Laptop|       5|    1200.0|2026-01-25|
|    1028|     Eli|  East|Monitor|       4|     220.0|2026-01-01|
|    1032|   Ishan|  East| Laptop|       3|    1200.0|2026-01-05|
|    1036|    Asha|  East| Laptop|       2|    1200.0|2026-01-09|
|    1040|     Eli|  East|Monitor|       1|     220.0|2026-01-13|
|    1044|   Ishan|  East| Laptop|       5|    1200.0|2026-01-17|
|    1048|    Asha|  East| Laptop|       4|    1200.0|2026-01-21|
|    1056|

In [0]:
typed.filter(F.col("quantity").between(2,5)).show(5)
typed.filter(F.col("product").isin("Laptop","Mouse")).show(5)
typed.filter(F.col("customer").like("A%")).show(5)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|
+--------+--------+------+-------+--------+----------+----------+
only showing top 5 rows
+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1006|    Gabe| North| Laptop|       2|    1200.

### combine conditions

In [0]:
typed.filter(
    (F.col("region")== "East") & (F.col("quantity") > 2)
).show(5)
typed.filter(
    (F.col("product")=="Laptop") | (F.col("product") == "Monitor")
).show(5)

typed.filter(~(F.col("region") =="West")).show(5)

#  & -> AND, | -> OR, ~ -> NOT

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|
|    1012|    Asha|  East| Laptop|       3|    1200.0|2026-01-13|
|    1024|    Asha|  East| Laptop|       5|    1200.0|2026-01-25|
|    1028|     Eli|  East|Monitor|       4|     220.0|2026-01-01|
+--------+--------+------+-------+--------+----------+----------+
only showing top 5 rows
+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.

### sorting

In [0]:
typed.orderBy(F.col("order_id")).show(5)
typed.orderBy(F.col("order_id").desc()).show(5)
typed.orderBy(F.col("order_id").desc(), F.col("quantity").asc()).show(5)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|
+--------+--------+------+-------+--------+----------+----------+
only showing top 5 rows
+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1060|    Asha|  East| Laptop|       1|    1200.0|2026-01-05|
|    1059|  Fatima|  West|Monitor|       5|     220.0|2026-01-04|
|    1058|   Kiran| North|Monitor|       4|     220.

### 2. when()/otherwise() and CASE WHEN

In [0]:
categorized = typed.withColumn(
    "order_size",
    F.when(F.col("quantity")>=3, "bulk")
    .when(F.col("quantity") == 2, "standard")
    .otherwise("small"))

categorized.show()

# using SQL, selectExpr()
typed.selectExpr(
     "*",
     "CASE WHEN quantity >= 3 THEN 'bulk' WHEN quantity = 2 THEN 'standard' ELSE 'small'  END AS order_Size"
).show()

+--------+--------+------+-------+--------+----------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|order_size|
+--------+--------+------+-------+--------+----------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|  standard|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|      bulk|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|      bulk|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|      bulk|
|    1005|    Lina| South|Monitor|       1|     220.0|2026-01-06|     small|
|    1006|    Gabe| North| Laptop|       2|    1200.0|2026-01-07|  standard|
|    1007|     Ben|  West| Laptop|       3|    1200.0|2026-01-08|      bulk|
|    1008|   Ishan|  East| Laptop|       4|    1200.0|2026-01-09|      bulk|
|    1009|   Divya| South|Monitor|       5|     220.0|2026-01-10|      bulk|
|    1010|   Kiran| North|Monitor|       1|     220.0|2026-01-11|     small|

# Data Cleaning and Common Function

### 1. Nulls : detect, fill, drop, coalesce

In [0]:
typed.filter(F.col('region').isNull()).show()
typed.filter(F.col('region').isNotNull()).show()

filled = typed.fillna({"region": "Unknown"})
filled.show(30)

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1013|    Hana|  NULL| Laptop|       4|    1200.0|2026-01-14|
|    1026|    Chen|  NULL| Laptop|       2|    1200.0|2026-01-27|
|    1039|     Jia|  NULL|Monitor|       5|     220.0|2026-01-12|
|    1052|     Eli|  NULL|Monitor|       3|     220.0|2026-01-25|
+--------+--------+------+-------+--------+----------+----------+

+--------+--------+------+-------+--------+----------+----------+
|order_id|customer|region|product|quantity|unit_price|order_date|
+--------+--------+------+-------+--------+----------+----------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|
|    1004|     Eli|  East|Monitor|       5|     220.0|2026-01-05|
|    1005

In [0]:

without_missing_region = typed.dropna(subset=["region"])
without_missing_region.count()

57

In [0]:
typed.withColumn(
    "region_clean",
    F.coalesce(F.col("region"),F.lit("Unknown"))
).select("region","region_clean").show()

+------+------------+
|region|region_clean|
+------+------------+
| South|       South|
| North|       North|
|  West|        West|
|  East|        East|
| South|       South|
| North|       North|
|  West|        West|
|  East|        East|
| South|       South|
| North|       North|
|  West|        West|
|  East|        East|
|  NULL|      Uknown|
| North|       North|
|  West|        West|
|  East|        East|
| South|       South|
| North|       North|
|  West|        West|
|  East|        East|
+------+------------+
only showing top 20 rows


### 2. Duplicates

In [0]:
typed.select("order_id").distinct().count()
typed_cleaned = typed.dropDuplicates()
typed_cleaned.count()


60

### 3. String Functions

In [0]:
string_demo = (
    typed
    .withColumn("customer_upper", F.upper("customer"))
    .withColumn("customer_lower", F.lower("customer"))
    .withColumn("customer_trimmed", F.trim("customer"))
    .withColumn("product_parts", F.split("product"," "))
    .withColumn("product_prefix", F.substring("product",1,3))
    .withColumn("product_clean", F.regexp_replace("product","[aeiou]","*"))
    .withColumn("customer_product", F.concat_ws(" - ","customer","product"))
)
string_demo.show()

+--------+--------+------+-------+--------+----------+----------+--------------+--------------+----------------+-------------+--------------+-------------+----------------+
|order_id|customer|region|product|quantity|unit_price|order_date|customer_upper|customer_lower|customer_trimmed|product_parts|product_prefix|product_clean|customer_product|
+--------+--------+------+-------+--------+----------+----------+--------------+--------------+----------------+-------------+--------------+-------------+----------------+
|    1001|    Hana| South| Laptop|       2|    1200.0|2026-01-02|          HANA|          hana|            Hana|     [Laptop]|           Lap|       L*pt*p|   Hana - Laptop|
|    1002|    Chen| North| Laptop|       3|    1200.0|2026-01-03|          CHEN|          chen|            Chen|     [Laptop]|           Lap|       L*pt*p|   Chen - Laptop|
|    1003|     Jia|  West|Monitor|       4|     220.0|2026-01-04|           JIA|           jia|             Jia|    [Monitor]|         

### 4. Numeric functions

In [0]:
typed.select(
    "unit_price",
    F.round("unit_price",1).alias("rounded_price"),
    F.abs(F.col("unit_price")-F.lit(100)).alias("distanse_from_100")).show(60)

+----------+-------------+-----------------+
|unit_price|rounded_price|distanse_from_100|
+----------+-------------+-----------------+
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|     220.0|        220.0|            120.0|
|    1200.0|       1200.0|           1100.0|
|    1200.0|       1200.0|           1100.0|
|    1200.

### 5. Date/Time functions

In [0]:
dates = typed.withColumn("order_date",F.to_date("order_date","yyyy-MM-dd"))

output = dates.select(
    "order_date",
    F.to_timestamp("order_date").alias("order_timestamps"),
    F.year("order_date").alias("year"),
    F.month("order_date").alias("month"),
    F.dayofmonth("order_date").alias("day"),
    F.dayofweek("order_date").alias("day_of_week"),
    F.dayofyear("order_date").alias("day_of_year"),
    F.quarter("order_date").alias("quarter"),
    F.date_add("order_date",7).alias("one weel later"),
    F.datediff(F.current_date(), "order_date").alias("days since order")
)
display(output)

order_date,order_timestamps,year,month,day,day_of_week,day_of_year,quarter,one weel later,days since order
2026-01-02,2026-01-02T00:00:00.000Z,2026,1,2,6,2,1,2026-01-09,308
2026-01-03,2026-01-03T00:00:00.000Z,2026,1,3,7,3,1,2026-01-10,307
2026-01-04,2026-01-04T00:00:00.000Z,2026,1,4,1,4,1,2026-01-11,306
2026-01-05,2026-01-05T00:00:00.000Z,2026,1,5,2,5,1,2026-01-12,305
2026-01-06,2026-01-06T00:00:00.000Z,2026,1,6,3,6,1,2026-01-13,304
2026-01-07,2026-01-07T00:00:00.000Z,2026,1,7,4,7,1,2026-01-14,303
2026-01-08,2026-01-08T00:00:00.000Z,2026,1,8,5,8,1,2026-01-15,302
2026-01-09,2026-01-09T00:00:00.000Z,2026,1,9,6,9,1,2026-01-16,301
2026-01-10,2026-01-10T00:00:00.000Z,2026,1,10,7,10,1,2026-01-17,300
2026-01-11,2026-01-11T00:00:00.000Z,2026,1,11,1,11,1,2026-01-18,299


# Aggregations

### 1. groupBY and agg()

In [0]:
by_region = typed.groupBy("region").agg(
    F.count("*").alias("row_count"),
    F.sum("quantity").alias("total_quantity"),
    F.avg("unit_price").alias("avg_price"),
    F.max("unit_price").alias("max_price"),
    F.min("unit_price").alias("min_price"),
    F.countDistinct("product").alias("product_count")
)

display(by_region)

region,row_count,total_quantity,avg_price,max_price,min_price,product_count
North,15,44,808.0,1200.0,220.0,2
East,14,42,920.0,1200.0,220.0,2
null,4,14,710.0,1200.0,220.0,2
South,14,41,500.0,1200.0,220.0,2
West,14,40,570.0,1200.0,220.0,2


In [0]:
typed.select(
    F.sum("quantity"),
    F.avg("unit_price"),
    F.max("unit_price")
).show()

+-------------+-----------------+---------------+
|sum(quantity)|  avg(unit_price)|max(unit_price)|
+-------------+-----------------+---------------+
|          181|701.9672131147541|         1200.0|
+-------------+-----------------+---------------+



### 2. condition aggregations

In [0]:
typed.groupBy("region").agg(
    F.sum(F.when(F.col("product") == "Laptop", F.col("quantity")).otherwise(0)).alias("laptop_units"),
    F.sum(F.when(F.col("product") == "Mouse", F.col("quantity")).otherwise(0)).alias("Mouse_units")
).show(60)

+------+------------+-----------+
|region|laptop_units|Mouse_units|
+------+------------+-----------+
| North|          28|          0|
|  East|          30|          0|
| South|          11|          0|
|  NULL|           6|          0|
|  West|          15|          0|
+------+------------+-----------+



### 3. pivot

In [0]:
display(typed.groupBy("region").pivot("product").sum("quantity"))

region,Laptop,Monitor
null,6,8
South,11,30
West,15,25
East,30,12
North,28,16


# Joins and Combining Dataframs

### 1. Prepare a customer lookup and join

In [0]:
inner_join = typed.join(customers, on = "customer", how= "inner")
display(inner_join)

customer,order_id,region,product,quantity,unit_price,order_date,customer_id,region,loyalty_tier,signup_date
Hana,1001,South,Laptop,2,1200.0,2026-01-02,2008,South,Silver,2024-08-15
Chen,1002,North,Laptop,3,1200.0,2026-01-03,2003,North,Bronze,2024-03-15
Jia,1003,West,Monitor,4,220.0,2026-01-04,2010,West,Gold,2024-10-15
Eli,1004,East,Monitor,5,220.0,2026-01-05,2005,East,Silver,2024-05-15
Lina,1005,South,Monitor,1,220.0,2026-01-06,2012,South,Bronze,2024-12-15
Gabe,1006,North,Laptop,2,1200.0,2026-01-07,2007,North,Gold,2024-07-15
Ben,1007,West,Laptop,3,1200.0,2026-01-08,2002,West,Silver,2024-02-15
Ishan,1008,East,Laptop,4,1200.0,2026-01-09,2009,East,Bronze,2024-09-15
Divya,1009,South,Monitor,5,220.0,2026-01-10,2004,South,Gold,2024-04-15
Kiran,1010,North,Monitor,1,220.0,2026-01-11,2011,North,Silver,2024-11-15


In [0]:
display(products)

product_id,product,unit_price,category
P101,Laptop,1200.0,Electronics
P102,Mouse,25.0,Accessories
P103,Keyboard,75.0,Accessories
P104,Monitor,220.0,Electronics
P105,Headphones,85.0,Accessories
P106,Webcam,60.0,Accessories


In [0]:
display(typed)

order_id,customer,region,product,quantity,unit_price,order_date
1001,Hana,South,Laptop,2,1200.0,2026-01-02
1002,Chen,North,Laptop,3,1200.0,2026-01-03
1003,Jia,West,Monitor,4,220.0,2026-01-04
1004,Eli,East,Monitor,5,220.0,2026-01-05
1005,Lina,South,Monitor,1,220.0,2026-01-06
1006,Gabe,North,Laptop,2,1200.0,2026-01-07
1007,Ben,West,Laptop,3,1200.0,2026-01-08
1008,Ishan,East,Laptop,4,1200.0,2026-01-09
1009,Divya,South,Monitor,5,220.0,2026-01-10
1010,Kiran,North,Monitor,1,220.0,2026-01-11


In [0]:
with_product_details = typed.join(products, on = "product", how = "left")
with_product_details.select("order_id","product","category","quantity",typed["unit_price"]).show()

+--------+-------+-----------+--------+----------+
|order_id|product|   category|quantity|unit_price|
+--------+-------+-----------+--------+----------+
|    1001| Laptop|Electronics|       2|    1200.0|
|    1002| Laptop|Electronics|       3|    1200.0|
|    1003|Monitor|Electronics|       4|     220.0|
|    1004|Monitor|Electronics|       5|     220.0|
|    1005|Monitor|Electronics|       1|     220.0|
|    1006| Laptop|Electronics|       2|    1200.0|
|    1007| Laptop|Electronics|       3|    1200.0|
|    1008| Laptop|Electronics|       4|    1200.0|
|    1009|Monitor|Electronics|       5|     220.0|
|    1010|Monitor|Electronics|       1|     220.0|
|    1011|Monitor|Electronics|       2|     220.0|
|    1012| Laptop|Electronics|       3|    1200.0|
|    1013| Laptop|Electronics|       4|    1200.0|
|    1014| Laptop|Electronics|       5|    1200.0|
|    1015|Monitor|Electronics|       1|     220.0|
|    1016|Monitor|Electronics|       2|     220.0|
|    1017|Monitor|Electronics| 

In [0]:
typed.join(customers, on="customer", how = "left").show()
typed.join(customers, on="customer", how = "full").show()
typed.join(customers, on="customer", how = "right").show()

+--------+--------+------+-------+--------+----------+----------+-----------+------+------------+-----------+
|customer|order_id|region|product|quantity|unit_price|order_date|customer_id|region|loyalty_tier|signup_date|
+--------+--------+------+-------+--------+----------+----------+-----------+------+------------+-----------+
|    Hana|    1001| South| Laptop|       2|    1200.0|2026-01-02|       2008| South|      Silver| 2024-08-15|
|    Chen|    1002| North| Laptop|       3|    1200.0|2026-01-03|       2003| North|      Bronze| 2024-03-15|
|     Jia|    1003|  West|Monitor|       4|     220.0|2026-01-04|       2010|  West|        Gold| 2024-10-15|
|     Eli|    1004|  East|Monitor|       5|     220.0|2026-01-05|       2005|  East|      Silver| 2024-05-15|
|    Lina|    1005| South|Monitor|       1|     220.0|2026-01-06|       2012| South|      Bronze| 2024-12-15|
|    Gabe|    1006| North| Laptop|       2|    1200.0|2026-01-07|       2007| North|        Gold| 2024-07-15|
|     Ben|

### 2. Multiple keys and duplicate column names

In [0]:
a = typed.alias("o")
b = products.alias("p")
with_product_details = a.join(b, on = "product", how = "left")
with_product_details.select(F.col("o.order_id"),F.col("o.product"),F.col("p.category"),F.col("o.quantity"),F.col("p.unit_price")).show()

+--------+-------+-----------+--------+----------+
|order_id|product|   category|quantity|unit_price|
+--------+-------+-----------+--------+----------+
|    1001| Laptop|Electronics|       2|    1200.0|
|    1002| Laptop|Electronics|       3|    1200.0|
|    1003|Monitor|Electronics|       4|     220.0|
|    1004|Monitor|Electronics|       5|     220.0|
|    1005|Monitor|Electronics|       1|     220.0|
|    1006| Laptop|Electronics|       2|    1200.0|
|    1007| Laptop|Electronics|       3|    1200.0|
|    1008| Laptop|Electronics|       4|    1200.0|
|    1009|Monitor|Electronics|       5|     220.0|
|    1010|Monitor|Electronics|       1|     220.0|
|    1011|Monitor|Electronics|       2|     220.0|
|    1012| Laptop|Electronics|       3|    1200.0|
|    1013| Laptop|Electronics|       4|    1200.0|
|    1014| Laptop|Electronics|       5|    1200.0|
|    1015|Monitor|Electronics|       1|     220.0|
|    1016|Monitor|Electronics|       2|     220.0|
|    1017|Monitor|Electronics| 

### 3.union() and unionByName()

In [0]:
jan = typed.select("order_id","customer","quantity")
feb = typed.select("order_id","customer","quantity")

# display(jan.union(feb))
display(jan.unionByName(feb))

order_id,customer,quantity
1001,Hana,2
1002,Chen,3
1003,Jia,4
1004,Eli,5
1005,Lina,1
1006,Gabe,2
1007,Ben,3
1008,Ishan,4
1009,Divya,5
1010,Kiran,1


# Window Functions

### 1. Define a window and rank rows

In [0]:
dated = typed.withColumn("order_date", F.to_date("order_date","yyyy-MM-dd"))

by_region_date = Window.partitionBy("region").orderBy(F.col("order_date").desc())

ranked = (
    dated
    .withColumn("row_Number", F.row_number().over(by_region_date))
    .withColumn("rank", F.rank().over(by_region_date))
    .withColumn("dense_rank", F.dense_rank().over(by_region_date))
    .withColumn("percent_rank", F.percent_rank().over(by_region_date))
)
display(ranked)

order_id,customer,region,product,quantity,unit_price,order_date,row_Number,rank,dense_rank,percent_rank
1026,Chen,null,Laptop,2,1200.0,2026-01-27,1,1,1,0.0
1052,Eli,null,Monitor,3,220.0,2026-01-25,2,2,2,0.3333333333333333
1013,Hana,null,Laptop,4,1200.0,2026-01-14,3,3,3,0.6666666666666666
1039,Jia,null,Monitor,5,220.0,2026-01-12,4,4,4,1.0
1024,Asha,East,Laptop,5,1200.0,2026-01-25,1,1,1,0.0
1020,Ishan,East,Laptop,1,1200.0,2026-01-21,2,2,2,0.07692307692307693
1048,Asha,East,Laptop,4,1200.0,2026-01-21,3,2,2,0.07692307692307693
1016,Eli,East,Monitor,2,220.0,2026-01-17,4,4,3,0.23076923076923078
1044,Ishan,East,Laptop,5,1200.0,2026-01-17,5,4,3,0.23076923076923078
1012,Asha,East,Laptop,3,1200.0,2026-01-13,6,6,4,0.38461538461538464


# User-Defined Function

In [0]:
from pyspark.sql.types import StringType
import pyspark.sql.functions as F
import random

kkey = "my_key" 

def normalize_product(value):
    if value is None:
        return None
    base_value = value.strip().lower()
    rand_val = random.random()
    
    return f"{base_value}_{kkey}_{rand_val}"

normalize_product_udf = F.udf(normalize_product, StringType())

with_udf = typed.withColumn("product_key", normalize_product_udf(F.col("product")))
display(with_udf)

order_id,customer,region,product,quantity,unit_price,order_date,product_key
1001,Hana,South,Laptop,2,1200.0,2026-01-02,laptop_my_key_0.6049988563507397
1002,Chen,North,Laptop,3,1200.0,2026-01-03,laptop_my_key_0.10754682979243269
1003,Jia,West,Monitor,4,220.0,2026-01-04,monitor_my_key_0.5255261329758802
1004,Eli,East,Monitor,5,220.0,2026-01-05,monitor_my_key_0.9034067530422878
1005,Lina,South,Monitor,1,220.0,2026-01-06,monitor_my_key_0.523288357156946
1006,Gabe,North,Laptop,2,1200.0,2026-01-07,laptop_my_key_0.6208074895625125
1007,Ben,West,Laptop,3,1200.0,2026-01-08,laptop_my_key_0.5346470026808939
1008,Ishan,East,Laptop,4,1200.0,2026-01-09,laptop_my_key_0.834710950267264
1009,Divya,South,Monitor,5,220.0,2026-01-10,monitor_my_key_0.930372330632048
1010,Kiran,North,Monitor,1,220.0,2026-01-11,monitor_my_key_0.4274559956314512


### 2.. register udf for SQL & temp views & spark sql

In [0]:
spark.udf.register("normalize_product_sql",normalize_product,StringType())
typed.createOrReplaceTempView("orders_view")

spark.sql(
    '''
        SELECT product, normalize_product_sql(product) AS product_key
        FROM orders_view
    '''
).show()

+-------+--------------------+
|product|         product_key|
+-------+--------------------+
| Laptop|laptop_my_key_0.0...|
| Laptop|laptop_my_key_0.9...|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
| Laptop|laptop_my_key_0.5...|
| Laptop|laptop_my_key_0.2...|
| Laptop|laptop_my_key_0.3...|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
| Laptop|laptop_my_key_0.5...|
| Laptop|laptop_my_key_0.2...|
| Laptop|laptop_my_key_0.7...|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
|Monitor|monitor_my_key_0....|
| Laptop|laptop_my_key_0.3...|
| Laptop|laptop_my_key_0.9...|
| Laptop|laptop_my_key_0.5...|
+-------+--------------------+
only showing top 20 rows
